<a href="https://colab.research.google.com/github/Khaledsaads/phi-2-fine-tuning/blob/main/phi2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
! pip install -U bitsandbytes>=0.46.1

In [2]:
! pip install trl

In [3]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig as bnb, TrainingArguments, Trainer
from trl import SFTTrainer
import torch
from datasets import load_dataset
import pandas as pd
from peft import LoraConfig, get_peft_model, TaskType

In [7]:
from huggingface_hub import interpreter_login
interpreter_login()


    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: HMGO-5ZWH

    Waiting for authorization...


In [5]:
dataset = load_dataset("arafatanam/Mental-Health-Couseling", split= 'train')

In [6]:
dataset

Dataset({
    features: ['Context', 'Response'],
    num_rows: 2752
})

In [7]:
df = pd.DataFrame(dataset)

In [9]:
df.head()

,Context,Response
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb..."
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see..."
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2752 entries, 0 to 2751
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Context   2752 non-null   object
 1   Response  2752 non-null   object
dtypes: object(2)
memory usage: 43.1+ KB


In [11]:
df.shape

(2752, 2)

In [12]:
df.describe()

,Context,Response
count,2752,2752
unique,995,2480
top,I have so many issues to address. I have a his...,
freq,47,4


In [13]:
from sklearn.model_selection import train_test_split
questions = pd.unique(df["Context"])
train_questions , val_questions = train_test_split(
    questions,
    test_size = 0.2,
    random_state=42
)
train_data = df[df["Context"].isin(train_questions)]
val_data = df[df["Context"].isin(val_questions)]

In [14]:
val_data.head()

,Context,Response
101,I have been diagnosed with general anxiety and...,"This can be a difficult situation. Typically,..."
102,I have been diagnosed with general anxiety and...,"At present, the American Disability Associatio..."
103,I have been diagnosed with general anxiety and...,Agree with the apartment manager's viewpoint t...
104,I have been diagnosed with general anxiety and...,This is tricky because there are different typ...
152,I am really worried about one of my friends be...,"First of all, I can tell that you really care ..."


In [15]:
def format_row(row):
  question = row["Context"]
  answer = row["Response"]
  formatted_text = f"[INST] {question} [/INST] {answer}"
  return formatted_text

In [16]:
train_data["formatted_text"]= train_data.apply(format_row, axis = 1)
val_data["formatted_text"]= val_data.apply(format_row, axis = 1)

/tmp/ipykernel_8600/1105972196.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_data["formatted_text"]= train_data.apply(format_row, axis = 1)
/tmp/ipykernel_8600/1105972196.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_data["formatted_text"]= val_data.apply(format_row, axis = 1)


In [17]:
train_data.head()

,Context,Response,formatted_text
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb...",[INST] I'm going through some things with my f...
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",[INST] I'm going through some things with my f...
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...,[INST] I'm going through some things with my f...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...,[INST] I'm going through some things with my f...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,[INST] I'm going through some things with my f...


In [18]:
val_data.head()

,Context,Response,formatted_text
101,I have been diagnosed with general anxiety and...,"This can be a difficult situation. Typically,...",[INST] I have been diagnosed with general anxi...
102,I have been diagnosed with general anxiety and...,"At present, the American Disability Associatio...",[INST] I have been diagnosed with general anxi...
103,I have been diagnosed with general anxiety and...,Agree with the apartment manager's viewpoint t...,[INST] I have been diagnosed with general anxi...
104,I have been diagnosed with general anxiety and...,This is tricky because there are different typ...,[INST] I have been diagnosed with general anxi...
152,I am really worried about one of my friends be...,"First of all, I can tell that you really care ...",[INST] I am really worried about one of my fri...


In [19]:
train_data.shape

(2196, 3)

In [20]:
train_data = train_data.rename(columns= {'formatted_text': 'Text'})
val_data = val_data.rename(columns= {'formatted_text': 'Text'})

In [21]:
train_data.head()

,Context,Response,Text
0,I'm going through some things with my feelings...,"If everyone thinks you're worthless, then mayb...",[INST] I'm going through some things with my f...
1,I'm going through some things with my feelings...,"Hello, and thank you for your question and see...",[INST] I'm going through some things with my f...
2,I'm going through some things with my feelings...,First thing I'd suggest is getting the sleep y...,[INST] I'm going through some things with my f...
3,I'm going through some things with my feelings...,Therapy is essential for those that are feelin...,[INST] I'm going through some things with my f...
4,I'm going through some things with my feelings...,I first want to let you know that you are not ...,[INST] I'm going through some things with my f...


In [22]:
new_train_df = train_data[["Text"]]
new_val_df = val_data[["Text"]]

In [23]:
new_train_df.head()

,Text
0,[INST] I'm going through some things with my f...
1,[INST] I'm going through some things with my f...
2,[INST] I'm going through some things with my f...
3,[INST] I'm going through some things with my f...
4,[INST] I'm going through some things with my f...


In [24]:
new_train_df.to_csv("formatted_train_data.csv", index = False)
new_val_df.to_csv("formatted_val_data.csv", index = False)

In [25]:
dataset = load_dataset(
    "csv",
    data_files={
        "train": "formatted_train_data.csv",
        "val": "formatted_val_data.csv"
    }
)
train_data = dataset['train']
val_data = dataset['val']

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

In [26]:
dataset

DatasetDict({
    train: Dataset({
        features: ['Text'],
        num_rows: 2196
    })
    val: Dataset({
        features: ['Text'],
        num_rows: 556
    })
})

## Split data

## Fine-Tuning

In [27]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig as bnb, TrainingArguments, Trainer
from trl import SFTTrainer
import torch
from datasets import load_dataset
import pandas as pd
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTConfig

base_model_checkpoint = "microsoft/phi-2"
new_model = "phi2-mental-health"
tokenizer = AutoTokenizer.from_pretrained(base_model_checkpoint)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

bnb_config = bnb(
    load_in_4bit = True,
    bnb_4bit_quant_type = 'nf4',
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = False
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_checkpoint,
    quantization_config = bnb_config,
    trust_remote_code = True,
    low_cpu_mem_usage =True,
    attn_implementation="sdpa",
    device_map= {"": 0})
model.config.use_cache = False

train_args = SFTConfig(
    output_dir="/mhGPT",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=32,
    eval_strategy="steps",
    eval_steps=34,
    save_steps=34,
    warmup_steps=5,
    logging_steps=5,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    max_length = 640,
    dataset_text_field = "Text",
    gradient_checkpointing=False,
    fp16=True, bf16=False
)
peft_config = LoraConfig(
    task_type = TaskType.CAUSAL_LM,
    r = 16,
    lora_alpha = 32,
    target_modules=["q_proj", "k_proj", "v_proj", "fc1", "fc2"],
    lora_dropout = 0.05,
    bias = 'none')
trainer = SFTTrainer(
    model = model,
    train_dataset = train_data,
    eval_dataset = val_data,
    peft_config = peft_config,
    args = train_args,
    processing_class = tokenizer,
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/2196 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2196 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (24360 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/2196 [00:00<?, ? examples/s]

...

Adding EOS to eval dataset:   0%|          | 0/556 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/556 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/556 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/556 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/556 [00:00<?, ? examples/s]

In [28]:
print(trainer.args.fp16, trainer.args.bf16, trainer.args.gradient_checkpointing)
print(model.config._attn_implementation)

True False False
sdpa


In [29]:
from collections import Counter

for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)

trainer.optimizer = None
trainer.lr_scheduler = None

print(Counter(p.dtype for p in trainer.model.parameters() if p.requires_grad))

Counter({torch.float32: 320})


In [30]:
trainer.train()

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
34,2.369518,2.326326,2.233909,620728.000000,0.469837
68,2.284562,2.313958,2.215785,1227986.000000,0.471752
70,2.319855,2.313949,2.215837,1251458.000000,0.471839


TrainOutput(global_step=70, training_loss=2.359559426988874, metrics={'train_runtime': 2979.424, 'train_samples_per_second': 1.474, 'train_steps_per_second': 0.023, 'total_flos': 2.586236831834112e+16, 'train_loss': 2.359559426988874, 'epoch': 2.0})

In [ ]:
trainer.save_model("mh-trainer")

In [36]:
model.push_to_hub("Khaledsaad1/mh-GPT")
tokenizer.push_to_hub("Khaledsaad1/mh-Tokenizer")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...u3bc1fj/model.safetensors:   0%|          | 4.68MB / 2.03GB            

Validating                    :           |   0%            

CommitInfo(commit_url='https://huggingface.co/Khaledsaad1/mh-Tokenizer/commit/26032573edc0069843a6c0c4e153bd391e92b082', commit_message='Upload tokenizer', commit_description='', oid='26032573edc0069843a6c0c4e153bd391e92b082', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Khaledsaad1/mh-Tokenizer', endpoint='https://huggingface.co', repo_type='model', repo_id='Khaledsaad1/mh-Tokenizer'), pr_revision=None, pr_num=None)

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()
repo_id = "Khaledsaad1/mh-trainer"
api = HfApi()
api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
api.upload_folder(
    repo_id=repo_id,
    repo_type="model",
    folder_path="/content/mh-trainer",
)